In [1]:
#importing libraries
import pandas as pd
import numpy as np
print("libraries succesfully imported")

libraries succesfully imported


In [2]:
#defining file path
call_file_path = r'../data/calls_sample.csv'
#list of columns to keep
cols_to_keep = ['Master_Incident_Number', 'Response_Date', 'Problem', 'Service_Area', 'Postal_Code']
#loading data with list of columns
df = pd.read_csv(call_file_path, usecols = cols_to_keep)

print("dataset loaded")
print(f'total rows: {len(df)}')

dataset loaded
total rows: 5455337


In [3]:
#dropping rows missing an incident number
df = df.dropna(subset=['Master_Incident_Number']).copy()

#standardizing postal codes to strings and cleaning unnecessary rows out
df['Postal_Code'] = df['Postal_Code'].astype(str).str.upper().str.strip()
exclude_list = ['NOT LISTED', 'OUT OF JURISDICTION']
df = df[~df['Postal_Code'].isin(exclude_list)].copy()

#removing decimals on zips
df['Postal_Code'] = df['Postal_Code'].str.replace('.0', '', regex=False).str[:5]
print("null incident numbers dropped & Zips cleaned")
print(f'total rows: {len(df)}')

null incident numbers dropped & Zips cleaned
total rows: 5275230


In [4]:
#converting Response_Date to datetime format
df['Response_Date'] = pd.to_datetime(df['Response_Date'], errors='coerce')
df = df.dropna(subset=['Response_Date']).copy()

#creating independent columns hour
df['Hour_Int']  = df['Response_Date'].dt.hour.astype(int)
df.head()

print("Hour column created")

Hour column created


In [5]:
#Grouping Problems into Groups
category_map = {}

#creating list of keywords for each group
keywords = {
    'Violent_Crime': [
        'Assault', 'Cutting', 'Robbery', 'Shooting', 'Rape', 'Threat',
        'Fired', 'Weapon', 'Fight', 'Family Violence', 'Sexual Offense', 'Wanted Person'
    ],
    'Property/Financial_Crime': [
        'Burglary', 'Theft', 'Shoplift', 'Mischief', 'Forgery', 'Stolen',
        'Property Found', 'Property Lost', 'Graffiti'
    ],
    'Public_Order': [
        'Disturbance', 'Drunk', 'Homeless', 'Prowler', 'Suspicious', 'Narcotic',
        'Ordinance','Lewd', 'Panhandle', 'Vagran', 'Loud', 'Visitation',
        'Protective Order', 'Liquor', 'Vice', 'Violation Sex Off Reg', 'Internet Predator'
    ],
    'Traffic_Safety': [
        'Accident', 'Traffic', 'DWI', 'Crash', 'Vehicle', 'Wrong Way'
    ],
    'Health_Welfare_Medical': [
        'Animal', 'Injured', 'Sick', 'Mental', 'Overdose', 'Suicide',
        'Welfare', 'Missing', 'DOA', 'Drowning'
    ],
    'Fire_Environmental_Hazard': [
        'Fire', 'Arson', 'Water', 'Unattended'
    ],
    'Security_Inspections_Alarms': [
        'Alarm', 'Inspection', 'ShotSpotter', 'Terminal'
    ],
    'Admin/Officer_Support': [
        'Assist', 'Officer', '911', 'Escort', 'SAPD', 'Prisoner',
        'Information', 'Patrol', 'Activity', 'Miscellaneous', 'CRT', 'Crime Plan',
        'On Site', 'SWAT', 'SAFD CALLTAKER ACA', 'Austin-Service Call', 'Move to Another Location'
    ]
}

#finding unique problems for mapping
unique_problems = df['Problem'].unique()

#assigning each problem into group
for problem in unique_problems:
    # Check each problem against our keywords
    for group, words in keywords.items():
        if any(word.lower() in str(problem).lower() for word in words):
            category_map[problem] = group
            break # Stop looking once we find a match

#Create Column
df['Problem_Group'] = df['Problem'].map(category_map).fillna('Other')

print("Group Column Created")

Group Column Created


In [8]:
#export as parquet
#parquet used to keep the data type assigned to column
output_path = r'E:/ROBBIE/School/PROJECT/GISCRIME/CLEAN/SATX_MASTER_FILE.parquet'
df.to_parquet(output_path, engine='pyarrow', index=False)

print(f"exported to {output_path}")
df.head()

exported to ../data/SATX_MASTER_FILE.parquet


,Master_Incident_Number,Response_Date,Problem,Service_Area,Postal_Code,Hour_Int,Problem_Group
0,SAPD-2025-1743255,2025-12-16 14:21:04,Homeless Camp,DOWNTOWN,78202,14,Public_Order
1,SAPD-2025-1797236,2025-12-27 18:53:22,Fight,CENTRAL,78207,18,Violent_Crime
2,SAPD-2025-1738820,2025-12-15 15:51:05,ID Theft,DOWNTOWN,78212,15,Property/Financial_Crime
3,SAPD-2025-1748643,2025-12-17 16:56:02,Mental Health In Progress,CENTRAL,78213,16,Health_Welfare_Medical
4,SAPD-2025-1790632,2025-12-26 10:42:43,Missing Person - Endangered,SOUTH,78223,10,Health_Welfare_Medical


Creating File with counts by Zip and Service Area

In [13]:
# 1. Load your cleaned data
df = pd.read_parquet(r'E:/ROBBIE/School/PROJECT/GISCRIME/CLEAN/SATX_MASTER_FILE.parquet')
print("file loaded")

file loaded


Master File For GIS by ZIP

In [28]:
#Binning Day and Night with Hour Int column
df['Shift'] = np.where((df['Hour_Int'] >= 6) & (df['Hour_Int'] < 18), 'Day', 'Night')

#Counts of how many times each 'Problem_Group' appears per ZIP
zip_crimes = df.pivot_table(
    index='Postal_Code', 
    columns='Problem_Group', 
    aggfunc='size', 
    fill_value=0
)

#Counts of Day vs Night per ZIP
zip_shifts = df.pivot_table(
    index='Postal_Code', 
    columns='Shift', 
    aggfunc='size', 
    fill_value=0
)

#Joining both tables
zip_master = pd.concat([zip_crimes, zip_shifts], axis=1).reset_index()

#Adding call total column
zip_master['Total_Calls'] = zip_master['Day'] + zip_master['Night']

#Keeping Service area in table for viewing
zip_to_sa = df.groupby('Postal_Code')['Service_Area'].agg(lambda x: x.value_counts().index[0]).reset_index()
zip_master = zip_master.merge(zip_to_sa, on='Postal_Code', how='left')

#Save to output folder
zip_master.to_parquet('E:/ROBBIE/School/PROJECT/GISCRIME/CLEAN/SATX_ZIP_Call_Master.parquet', index=False)

print("Master File Created Successfully!")

Master File Created Successfully!


Master File For GIS by Service Area

In [29]:
#Binning Day and Night with Hour Int column
df['Shift'] = np.where((df['Hour_Int'] >= 6) & (df['Hour_Int'] < 18), 'Day', 'Night')

#Counts of how many times each 'Problem_Group' appears per Area
area_call = df.pivot_table(
    index='Service_Area', 
    columns='Problem_Group', 
    aggfunc='size', 
    fill_value=0
)

#Counts of Day vs Night per Area
area_shifts = df.pivot_table(
    index='Service_Area', 
    columns='Shift', 
    aggfunc='size', 
    fill_value=0
)

#Joining both tables
area_master = pd.concat([area_call, area_shifts], axis=1).reset_index()

#Adding call total column
area_master['Total_Calls'] = area_master['Day'] + area_master['Night']

#Keeping ZIP in table for viewing
area_to_sa = df.groupby('Service_Area')['Postal_Code'].agg(lambda x: x.value_counts().index[0]).reset_index()
area_master = area_master.merge(area_to_sa, on='Service_Area', how='left')

#Save to output folder
area_master.to_parquet('E:/ROBBIE/School/PROJECT/GISCRIME/CLEAN/SATX_Area_Call_Master.parquet', index=False)

print("Master File Created Successfully!")

Master File Created Successfully!
